# Personal CV Website Capstone Project

**Student:** Mewcha Amha Gebremedhin  
**Project type:** Option F — My own idea: generate and publish a personal CV website with Python.

This notebook follows the FGCU Dendritic AI Summer Academy capstone requirements. It uses a JSON data file, reads and validates the data with error handling, generates a static HTML/CSS website, and includes visible Copilot prompt comments above AI-assisted code cells.

## 1. Project Requirements Checklist

- At least 3 AI-assisted Python code cells with clear functions or logical blocks
- At least 1 data operation using CSV or JSON
- Error handling for file reads and inputs
- Docstrings for every function
- At least 2 visible `# COPILOT PROMPT:` comments
- Reflection answers included in the notebook
- Website files prepared for GitHub Pages publishing

## 2. Project Plan

The goal is to build a professional personal website from structured CV information.

The workflow is:

1. Load my CV information from `cv_data.json`.
2. Validate that the required fields are present.
3. Generate `docs/index.html`.
4. Generate `docs/style.css`.
5. Copy or link the CV PDF.
6. Preview the generated files.
7. Publish the `docs/` folder using GitHub Pages.

## 3. Import Libraries and Set File Paths

In [1]:
# POSSIBLE PROMPTS LIST:
# - Act as a Python developer. Create a clean setup cell for a notebook that imports only standard libraries.
# - Define reusable pathlib paths for a JSON input file and GitHub Pages docs output folder.
# - Add comments explaining why each path is needed for a personal website generator.

from pathlib import Path
import json
import html
import shutil
from typing import Any, Dict, List

PROJECT_ROOT = Path.cwd()
DATA_FILE = PROJECT_ROOT / "cv_data.json"
OUTPUT_DIR = PROJECT_ROOT / "docs"
HTML_FILE = OUTPUT_DIR / "index.html"
CSS_FILE = OUTPUT_DIR / "style.css"
CV_PDF_SOURCE = PROJECT_ROOT / "Mewcha_A_Gebremedhin_CV.pdf"
CV_PDF_OUTPUT = OUTPUT_DIR / "Mewcha_A_Gebremedhin_CV.pdf"

OUTPUT_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data file:", DATA_FILE)
print("Website output folder:", OUTPUT_DIR)

Project root: /mnt/data/Gebremedhin_Mewcha_Capstone_Website
Data file: /mnt/data/Gebremedhin_Mewcha_Capstone_Website/cv_data.json
Website output folder: /mnt/data/Gebremedhin_Mewcha_Capstone_Website/docs


## 4. Load and Validate the CV JSON Data

This cell performs the required JSON data operation. It includes error handling so the notebook can run to completion even if the file is missing or has invalid JSON.

In [2]:
# COPILOT PROMPT: Act as a careful Python data engineer. Write a function that loads a JSON file,
# handles missing files and invalid JSON without crashing, and returns a dictionary of CV data.
# POSSIBLE PROMPTS LIST:
# - Add type hints and a docstring to the JSON loading function.
# - If the file is missing, return an empty dictionary and print a helpful message.
# - If the JSON is invalid, catch the decode error and explain what happened.

def load_cv_data(filepath: Path) -> Dict[str, Any]:
    """Load CV data from a JSON file.

    Parameters:
        filepath: Path to the JSON file that stores structured CV information.

    Returns:
        A dictionary containing CV data. If the file is missing or invalid, an empty dictionary is returned.
    """
    try:
        with filepath.open("r", encoding="utf-8") as file:
            data = json.load(file)
        print(f"Loaded CV data from {filepath.name}.")
        return data
    except FileNotFoundError:
        print(f"Warning: {filepath} was not found. Please include cv_data.json with your submission.")
        return {}
    except json.JSONDecodeError as error:
        print(f"Warning: {filepath} contains invalid JSON: {error}")
        return {}


# COPILOT PROMPT: Act as a code reviewer. Create a validation function that checks required
# website fields and reports missing fields without raising an unhandled exception.
def validate_cv_data(data: Dict[str, Any]) -> bool:
    """Validate that the CV data contains the minimum fields needed to build the website.

    Parameters:
        data: Dictionary containing structured CV information.

    Returns:
        True if all required fields are available, otherwise False.
    """
    required_fields = ["name", "title", "email", "summary", "skills", "education", "experience"]
    missing_fields = [field for field in required_fields if not data.get(field)]

    if missing_fields:
        print("Missing required fields:", ", ".join(missing_fields))
        return False

    print("CV data passed validation.")
    return True


cv_data = load_cv_data(DATA_FILE)
data_is_valid = validate_cv_data(cv_data)

Loaded CV data from cv_data.json.
CV data passed validation.


## 5. Generate the Website HTML

This cell converts structured CV data into a professional one-page website. It escapes text before writing HTML to reduce the risk of malformed or unsafe output.

In [3]:
# COPILOT PROMPT: Act as a front-end developer. Generate semantic HTML for a personal academic CV website
# using structured JSON data, with sections for About, Skills, Education, Experience, Research, and Contact.
# POSSIBLE PROMPTS LIST:
# - Use accessible section headings and simple navigation links.
# - Escape text values before inserting them into HTML.
# - Include a link to the downloadable CV PDF if it exists in the docs folder.

def safe_text(value: Any) -> str:
    """Convert a value to escaped HTML-safe text.

    Parameters:
        value: Any Python value that should be displayed in HTML.

    Returns:
        A string escaped for safe insertion into HTML content.
    """
    return html.escape(str(value))


def build_link_row(data: Dict[str, Any]) -> str:
    """Build a row of contact and profile links for the website hero section.

    Parameters:
        data: Dictionary containing email, ORCID, LinkedIn, and related profile data.

    Returns:
        A string containing HTML anchor elements for available links.
    """
    links = []

    if data.get("email"):
        links.append(f'<a href="mailto:{safe_text(data["email"])}">Email</a>')
    if data.get("orcid"):
        links.append(f'<a href="{safe_text(data["orcid"])}" target="_blank" rel="noopener">ORCID</a>')
    if data.get("linkedin"):
        links.append(f'<a href="{safe_text(data["linkedin"])}" target="_blank" rel="noopener">LinkedIn</a>')

    links.append('<a href="Mewcha_A_Gebremedhin_CV.pdf" target="_blank" rel="noopener">Download CV</a>')
    return "".join(links)


def build_website_html(data: Dict[str, Any]) -> str:
    """Create the full HTML document for the personal CV website.

    Parameters:
        data: Dictionary containing structured CV content.

    Returns:
        A complete HTML document as a string.
    """
    skills_html = "".join(f"<li>{safe_text(skill)}</li>" for skill in data.get("skills", []))

    education_html = "".join(
        f'<article class="card"><h3>{safe_text(item.get("degree", ""))}</h3>'
        f'<p><strong>{safe_text(item.get("years", ""))}</strong> · {safe_text(item.get("institution", ""))}</p></article>'
        for item in data.get("education", [])
    )

    experience_html = "".join(
        f'<article class="timeline-item"><h3>{safe_text(item.get("role", ""))}</h3>'
        f'<p><strong>{safe_text(item.get("years", ""))}</strong> · {safe_text(item.get("organization", ""))}</p></article>'
        for item in data.get("experience", [])
    )

    project_html = "".join(
        f'<article class="card"><h3>{safe_text(project.get("name", ""))}</h3>'
        f'<p>{safe_text(project.get("description", ""))}</p></article>'
        for project in data.get("projects", [])
    )

    publication_html = "".join(
        f"<li>{safe_text(publication)}</li>"
        for publication in data.get("selected_publications", [])
    )

    award_html = "".join(
        f"<li>{safe_text(award)}</li>"
        for award in data.get("awards_service", [])
    )

    return f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>{safe_text(data.get("name", "Personal Website"))}</title>
  <link rel="stylesheet" href="style.css" />
</head>
<body>
  <header class="hero">
    <nav class="topnav" aria-label="Main navigation">
      <a href="#about">About</a>
      <a href="#skills">Skills</a>
      <a href="#experience">Experience</a>
      <a href="#research">Research</a>
      <a href="#contact">Contact</a>
    </nav>
    <div class="hero-content">
      <p class="eyebrow">Personal CV Website</p>
      <h1>{safe_text(data.get("name", ""))}</h1>
      <h2>{safe_text(data.get("title", ""))}</h2>
      <p class="hero-summary">{safe_text(data.get("summary", ""))}</p>
      <div class="button-row">{build_link_row(data)}</div>
    </div>
  </header>

  <main>
    <section id="about" class="section">
      <h2>About Me</h2>
      <p>{safe_text(data.get("summary", ""))}</p>
    </section>

    <section id="skills" class="section">
      <h2>Core Skills</h2>
      <ul class="skill-grid">{skills_html}</ul>
    </section>

    <section id="education" class="section">
      <h2>Education</h2>
      <div class="grid">{education_html}</div>
    </section>

    <section id="experience" class="section">
      <h2>Professional Experience</h2>
      <div class="timeline">{experience_html}</div>
    </section>

    <section id="research" class="section">
      <h2>Selected Research Projects</h2>
      <div class="grid">{project_html}</div>
      <h3>Selected Publications</h3>
      <ol>{publication_html}</ol>
    </section>

    <section id="service" class="section">
      <h2>Awards and Professional Service</h2>
      <ul>{award_html}</ul>
    </section>

    <section id="contact" class="section contact">
      <h2>Contact</h2>
      <p>Email: <a href="mailto:{safe_text(data.get("email", ""))}">{safe_text(data.get("email", ""))}</a></p>
      <p>Location: {safe_text(data.get("location", ""))}</p>
    </section>
  </main>

  <footer>
    <p>© 2026 {safe_text(data.get("name", ""))}. Built with Python, Jupyter Notebook, GitHub Copilot prompts, and GitHub Pages.</p>
  </footer>
</body>
</html>"""

## 6. Generate CSS Styling

This cell writes a separate CSS file so the website is easier to maintain and publish on GitHub Pages.

In [4]:
# COPILOT PROMPT: Act as a web designer. Write clean CSS for a responsive academic portfolio website
# with a professional hero section, cards, readable spacing, and mobile-friendly layout.
# POSSIBLE PROMPTS LIST:
# - Use CSS grid for skills, education, and research projects.
# - Make the navigation links simple and accessible.
# - Keep the design professional for an academic CV website.

def build_css() -> str:
    """Create the CSS stylesheet for the generated personal website.

    Parameters:
        None.

    Returns:
        A string containing CSS rules for layout, typography, navigation, cards, and responsive design.
    """
    return """* { box-sizing: border-box; }
body {
  margin: 0;
  font-family: Arial, Helvetica, sans-serif;
  color: #1f2937;
  background: #f8fafc;
  line-height: 1.6;
}
a { color: #075985; }
.hero {
  min-height: 90vh;
  color: white;
  background: linear-gradient(135deg, #0f172a, #0e7490);
  padding: 2rem;
  display: flex;
  flex-direction: column;
}
.topnav {
  display: flex;
  justify-content: center;
  flex-wrap: wrap;
  gap: 1rem;
}
.topnav a {
  color: white;
  text-decoration: none;
  font-weight: bold;
}
.hero-content {
  max-width: 1000px;
  margin: auto;
  text-align: center;
}
.eyebrow {
  text-transform: uppercase;
  letter-spacing: 0.2em;
  font-size: 0.85rem;
  color: #bae6fd;
}
h1 {
  font-size: clamp(2.5rem, 8vw, 5rem);
  line-height: 1;
  margin: 0.5rem 0;
}
.hero h2 {
  color: #e0f2fe;
  font-weight: 500;
}
.hero-summary {
  max-width: 850px;
  margin: 1.5rem auto;
  font-size: 1.1rem;
}
.button-row {
  display: flex;
  gap: 0.8rem;
  justify-content: center;
  flex-wrap: wrap;
}
.button-row a {
  color: #0f172a;
  background: white;
  border-radius: 999px;
  padding: 0.75rem 1rem;
  text-decoration: none;
  font-weight: bold;
}
.section {
  max-width: 1100px;
  margin: 0 auto;
  padding: 4rem 1.5rem;
}
.section h2 {
  font-size: 2rem;
  color: #0f172a;
  border-bottom: 4px solid #0e7490;
  display: inline-block;
  padding-bottom: 0.25rem;
}
.grid, .skill-grid {
  display: grid;
  grid-template-columns: repeat(auto-fit, minmax(260px, 1fr));
  gap: 1rem;
}
.card, .timeline-item, .skill-grid li {
  background: white;
  border-radius: 1rem;
  padding: 1.25rem;
  box-shadow: 0 10px 30px rgba(15, 23, 42, 0.08);
}
.skill-grid {
  list-style: none;
  padding: 0;
}
.timeline {
  border-left: 4px solid #0e7490;
  padding-left: 1rem;
}
.timeline-item {
  margin-bottom: 1rem;
}
.contact {
  background: white;
  border-radius: 1rem;
  margin-bottom: 2rem;
}
footer {
  text-align: center;
  padding: 2rem;
  background: #0f172a;
  color: white;
}
"""


print("CSS function is ready.")

CSS function is ready.


## 7. Write Website Files and Copy CV PDF

This cell writes the final website files to the `docs/` folder. GitHub Pages can publish directly from this folder.

In [5]:
# COPILOT PROMPT: Act as a deployment assistant. Write a function that creates the docs folder,
# writes index.html and style.css, copies the CV PDF when available, and handles errors gracefully.
# POSSIBLE PROMPTS LIST:
# - Print clear success messages for each generated file.
# - Do not crash if the CV PDF is missing.
# - Return a list of generated files for final checking.

def write_site_files(data: Dict[str, Any], output_dir: Path) -> List[Path]:
    """Write the generated website HTML, CSS, and optional CV PDF into an output folder.

    Parameters:
        data: Dictionary containing structured CV data.
        output_dir: Folder where GitHub Pages-ready website files should be written.

    Returns:
        A list of Path objects for files successfully created or copied.
    """
    generated_files = []

    try:
        output_dir.mkdir(exist_ok=True)
        HTML_FILE.write_text(build_website_html(data), encoding="utf-8")
        CSS_FILE.write_text(build_css(), encoding="utf-8")
        generated_files.extend([HTML_FILE, CSS_FILE])
        print(f"Wrote {HTML_FILE}")
        print(f"Wrote {CSS_FILE}")

        if CV_PDF_SOURCE.exists():
            shutil.copy(CV_PDF_SOURCE, CV_PDF_OUTPUT)
            generated_files.append(CV_PDF_OUTPUT)
            print(f"Copied CV PDF to {CV_PDF_OUTPUT}")
        else:
            print("CV PDF was not found in the project root. The website will still run.")

    except OSError as error:
        print(f"Could not write website files: {error}")

    return generated_files


if data_is_valid:
    generated_files = write_site_files(cv_data, OUTPUT_DIR)
else:
    generated_files = []
    print("Website files were not generated because the data did not pass validation.")

generated_files

Wrote /mnt/data/Gebremedhin_Mewcha_Capstone_Website/docs/index.html
Wrote /mnt/data/Gebremedhin_Mewcha_Capstone_Website/docs/style.css
Copied CV PDF to /mnt/data/Gebremedhin_Mewcha_Capstone_Website/docs/Mewcha_A_Gebremedhin_CV.pdf


[PosixPath('/mnt/data/Gebremedhin_Mewcha_Capstone_Website/docs/index.html'),
 PosixPath('/mnt/data/Gebremedhin_Mewcha_Capstone_Website/docs/style.css'),
 PosixPath('/mnt/data/Gebremedhin_Mewcha_Capstone_Website/docs/Mewcha_A_Gebremedhin_CV.pdf')]

## 8. Create a README File for GitHub and Canvas

The README explains how to run the notebook and publish the generated website.

In [6]:
# COPILOT PROMPT: Act as a documentation writer. Generate a README file explaining the capstone project,
# the file structure, how to run the notebook, and how to publish the website with GitHub Pages.
# POSSIBLE PROMPTS LIST:
# - Include exact GitHub Pages settings for publishing from the docs folder.
# - Mention that cv_data.json must be submitted with the notebook.
# - Keep the README clear for an instructor who will run the project.

def write_readme(readme_path: Path) -> Path:
    """Write a README file describing the capstone project and publishing steps.

    Parameters:
        readme_path: Path where the README text file should be written.

    Returns:
        The Path to the README file that was written.
    """
    readme_text = """# Personal CV Website Capstone

This capstone uses Python in a Jupyter Notebook to generate a personal CV website from a JSON data file.

## Files to submit

- Gebremedhin_Mewcha_Capstone.ipynb
- cv_data.json
- README.txt
- The docs folder, if your instructor wants to review the generated website files

## How to run

1. Open the notebook in VS Code or Jupyter.
2. Make sure cv_data.json is in the same folder as the notebook.
3. Click Kernel -> Restart -> Run All Cells.
4. Confirm that docs/index.html and docs/style.css are generated.

## How to publish on GitHub Pages

1. Create a new GitHub repository.
2. Upload this notebook, cv_data.json, README.txt, and the docs folder.
3. Open the repository Settings tab.
4. Go to Pages.
5. Under Build and deployment, select Deploy from a branch.
6. Select the main branch and the /docs folder.
7. Save and wait for GitHub to publish the site.
"""
    readme_path.write_text(readme_text, encoding="utf-8")
    print(f"Wrote {readme_path}")
    return readme_path


readme_file = write_readme(PROJECT_ROOT / "README.txt")
readme_file

Wrote /mnt/data/Gebremedhin_Mewcha_Capstone_Website/README.txt


PosixPath('/mnt/data/Gebremedhin_Mewcha_Capstone_Website/README.txt')

## 9. Final File Check

This cell checks that the expected project files exist. It does not raise an unhandled exception, so the notebook can still run to completion.

In [7]:
# POSSIBLE PROMPTS LIST:
# - Act as a QA tester and verify that all expected capstone files exist.
# - Print a friendly checklist of files that are ready and files that are missing.
# - Avoid raising exceptions because the notebook must run all cells successfully.

def check_expected_files(expected_files: List[Path]) -> None:
    """Print a checklist showing whether expected project files exist.

    Parameters:
        expected_files: List of files that should exist after the notebook runs.

    Returns:
        None. The function prints a status report.
    """
    print("Final project file check:")
    for file_path in expected_files:
        status = "FOUND" if file_path.exists() else "MISSING"
        print(f"{status}: {file_path}")


check_expected_files([
    DATA_FILE,
    HTML_FILE,
    CSS_FILE,
    PROJECT_ROOT / "README.txt",
])

Final project file check:
FOUND: /mnt/data/Gebremedhin_Mewcha_Capstone_Website/cv_data.json
FOUND: /mnt/data/Gebremedhin_Mewcha_Capstone_Website/docs/index.html
FOUND: /mnt/data/Gebremedhin_Mewcha_Capstone_Website/docs/style.css
FOUND: /mnt/data/Gebremedhin_Mewcha_Capstone_Website/README.txt


## 10. GitHub Publishing Procedure

Follow these steps after the notebook runs successfully.

### Procedure A — Prepare the Repository

1. Go to GitHub and sign in.
2. Create a new repository. A good name is `personal-cv-website` or `mewcha-gebremedhin.github.io`.
3. Upload the following files and folders:
   - `Gebremedhin_Mewcha_Capstone.ipynb`
   - `cv_data.json`
   - `README.txt`
   - `docs/index.html`
   - `docs/style.css`
   - `docs/Mewcha_A_Gebremedhin_CV.pdf`, if available

### Procedure B — Enable GitHub Pages

1. Open the repository.
2. Click **Settings**.
3. Click **Pages** in the left sidebar.
4. Under **Build and deployment**, choose **Deploy from a branch**.
5. Select branch: **main**.
6. Select folder: **/docs**.
7. Click **Save**.
8. Wait for GitHub to publish the website.

### Procedure C — Submit to Canvas

Submit the notebook and the data file required by the notebook:

- `Gebremedhin_Mewcha_Capstone.ipynb`
- `cv_data.json`
- `README.txt`

If allowed, also submit the zipped website folder for convenience.

## 11. Reflection Sheet

### Question 1 — Describe your two most effective Copilot prompts.

**Prompt 1 text:**  
Act as a careful Python data engineer. Write a function that loads a JSON file, handles missing files and invalid JSON without crashing, and returns a dictionary of CV data.

**Why it worked:**  
This prompt worked because it used a role-based technique by asking Copilot to act as a Python data engineer. It also included constraints: handle missing files, handle invalid JSON, and return a dictionary. These constraints produced safer code than a simple prompt like “load my JSON file” because the final function protects the notebook from unhandled exceptions.

**Prompt 2 text:**  
Act as a front-end developer. Generate semantic HTML for a personal academic CV website using structured JSON data, with sections for About, Skills, Education, Experience, Research, and Contact.

**Why it worked:**  
This prompt worked because it clearly described the role, the input data type, and the required website sections. It produced a complete structure instead of a small HTML fragment. The prompt also helped keep the website focused on an academic and professional CV audience.

### Question 2 — What was the hardest bug you faced, and how did AI help you fix it?

The hardest bug was making sure the notebook would still run if a file was missing or if the JSON file was formatted incorrectly. Without error handling, the instructor could restart the kernel and run all cells, but the notebook might stop before reaching the website generation step. AI helped by suggesting `try`, `except FileNotFoundError`, and `except json.JSONDecodeError` blocks. I learned that a capstone notebook should fail gracefully and print helpful messages instead of stopping with an unhandled exception.

### Question 3 — Ethical Reflection: identify one AI coding risk relevant to your project.

One AI coding risk in this project is privacy. A CV website can accidentally publish personal contact details, employment history, or other information that the owner did not intend to make public. I addressed this by keeping the data in a clear `cv_data.json` file so the information can be reviewed before publishing. Before making the repository public, I would re-check the CV PDF and website text to make sure only appropriate professional information is included.

### Question 4 — What would you build next using AI-assisted coding?

Next, I would build an interactive hydrology data dashboard that visualizes rainfall, evapotranspiration, groundwater levels, and salinity trends. The dashboard could use Python to clean datasets and generate graphs, then use a web interface for public communication. This would support research, teaching, and community outreach by making environmental data easier to understand. AI-assisted coding would help me prototype the dashboard faster and improve the code structure.